# Cross Validation

With SVC as the chosen model

In [ ]:
import pandas as pd
from datetime import *
import os 
import sys  
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report,roc_curve,roc_auc_score, auc, cohen_kappa_score
import numpy as np
import matplotlib.pyplot as plt
from autofeat import autofeat
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,label_binarize
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from numpy import interp
from itertools import cycle
import seaborn as sns

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF

RAWDATAFILES={}
scores = ["precision", "recall"]  # we optimize for precision, since we let the user classify if unsure


pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
FEATURES_DF = pd.read_pickle(os.path.join(PREDICTIONS_Data_DIR,"features.pkl"))
EXERCISE_DF = pd.read_pickle(os.path.join(PREDICTIONS_Data_DIR,"exercise_dict.pkl"))

exercise_tuple = [tup for tup in EXERCISE_DF.set_index("key").itertuples()]
print(exercise_tuple)

feature_cols = FEATURES_DF.columns[:-3]
target_cols = FEATURES_DF.columns[-3:]

print(feature_cols)
print(target_cols)

for col in feature_cols:
    FEATURES_DF[col] = FEATURES_DF[col].astype(float)

FEATURES_DF["target"]= pd.Series(FEATURES_DF["target"], dtype=pd.Int64Dtype())

In [ ]:
mask =((FEATURES_DF.target>1) & (FEATURES_DF.target.isin(list(EXERCISE_DF[EXERCISE_DF.target_count>40].index))))

df = FEATURES_DF[mask].copy()
print("classes used: ")
display(df.target.value_counts(dropna=False))

X_train, X_test, y_train, y_test = train_test_split(df[feature_cols], df.target, random_state=12,stratify=df.target, test_size=0.2,shuffle=True) # stratify preserves the class ratios
scaler = StandardScaler() #since our data has different kind of units (HPBPM), gravity, acceleration and angles, scaling is actually not necessary
X_train_scaled  = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(set(y_test) - set(y_train))


In [ ]:
#### FS
svc = SVC(class_weight="balanced")
pipe_scaler = StandardScaler()
pipeline = Pipeline([('transformer',pipe_scaler),('estimator', svc)])
param_grid = {
    'estimator__kernel': ["linear", "poly", "rbf","sigmoid"],
    'estimator__degree': [2,3],
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
    }
# tuned_parameters = [
#     {"estimator__kernel": ["rbf"], "estimator__gamma": [1e-3, 1e-4], "estimator__C": [1, 10, 100, 1000]},
#     {"estimator__kernel": ["poly"], "estimator__gamma": [1e-3, 1e-4,1e-2, 1e3,1e2], "estimator__C": [1, 10, 100, 1000], "estimator__degree": [2,3]},
#     {"estimator__kernel": ["linear"], "estimator__C": [1, 10, 100, 1000], "estimator__gamma": [0.001, 0.01, 0.1, 1, 10, 100]}
# ]
grid_search1 = GridSearchCV(pipeline, param_grid=param_grid, cv=8,n_jobs=24) # search using pipeline
grid_search1.fit(X_train, y_train)
print("Best parameters - with pipeline: {}".format(grid_search1.best_params_))
print("Best cross-validation score - with pipeline: {:.2f}".format(grid_search1.best_score_))
#### FS

In [ ]:
#### vs PCS
pca_scaler = StandardScaler()
X_scaled = pca_scaler.fit_transform(df[feature_cols])
pca = PCA(.9)
X_pca = pca.fit_transform(X_scaled)
#X_pca = pca.fit_transform(df[feature_cols])

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, df.target,  random_state=42,stratify=df.target, test_size=0.2,shuffle=True) # stratify preserves the class ratios

svc = SVC(class_weight="balanced")
pipeline2 = Pipeline([('estimator', svc)])
param_grid = {
    'estimator__kernel': ["linear", "poly", "rbf","sigmoid"],
    'estimator__degree': [2,3],
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10, 100]
    }
grid_search2 = GridSearchCV(pipeline2, param_grid, cv=8,n_jobs=24) # optimize for precision, since we let the user classify if unsure
grid_search2.fit(X_train_pca, y_train_pca)
print("Best parameters - with pipeline: {}".format(grid_search2.best_params_))
print("Best cross-validation score - with pipeline: {:.2f}".format(grid_search2.best_score_))
#### PCS

In [ ]:
# Since Feature Space provide better scoring, than Principal Component Space, we narrow down to the best c and gamma in FS
best_score = 0

# using pipeline to scale data for CV
for i_gamma in [0.004,0.001,0.0008]: #[0.005]: #apart from 0.01, [0.02,0.005], [0.008,0.006,0.004]:
  for i_C in [78,80,84]: #[76]: #apart from 100, [50, 76, 80, 120 , 150]: [75,85] [50, 76, 80, 120 , 150]:
    # Multiple metric evaluation using cross_validate
    svc = SVC(class_weight="balanced",kernel="rbf",gamma=i_gamma,C=i_C)
    pipe_scaler = StandardScaler()
    pipeline = Pipeline([('transformer',pipe_scaler),('estimator', svc)])
    scores = cross_validate(svc, X_train, y_train, cv=5, scoring=('r2', 'precision_weighted'), return_train_score=True,)

    score = np.mean(scores["test_precision_weighted"]) #optimize for Precision

    # if we got a better score, store the score and parameters
    if score > best_score:
      best_score = score
      best_parameters = {'C': i_C, 'gamma': i_gamma}
# rebuild a model on the combined training and validation set
svm = SVC(**best_parameters)
pipeline = Pipeline([('transformer', scaler), ('estimator', svm)])

pipeline.fit(X_train, y_train)

# using only test set to evaluate
test_score = pipeline.score(X_test, y_test)
print("Best score on validation set: {:.2f}".format(best_score))
print("Best parameters: ", best_parameters)
print("Test set score with best parameters: {:.2f}".format(test_score))

In [ ]:
from sklearn.model_selection import cross_validate

svc = SVC(kernel="rbf",class_weight="balanced",C=80,gamma=0.001)
scaler = StandardScaler()
pipeline = Pipeline([('transformer', scaler), ('estimator', svc)])

search = GridSearchCV(pipeline, param_grid={}, cv=10)

scores = cross_validate(search, df[feature_cols], df.target, cv=5,
                        scoring=('r2', 'precision_weighted'),
                        return_train_score=True)


print("Cross-validation scores R2 train: {:.2f}".format(scores["train_r2"].mean()))
print("Cross-validation scores R2 test: {:.2f}".format(scores["test_r2"].mean()))
print("Cross-validation scores precision weighted train: {:.2f}".format(scores["train_precision_weighted"].mean()))
print("Cross-validation scores precision weighted test: {:.2f}".format(scores["test_precision_weighted"].mean()))

In [ ]:
import seaborn as sns
svc = SVC(kernel="rbf",class_weight="balanced",C=80,gamma=0.001)
svc.fit(X_train_scaled,y_train)
y_pred = svc.predict(X_test_scaled)
confusion = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n{}".format(confusion))

def plotCM(y_Truth,y_Predicted,labels,clf_classes,title):
    cm = confusion_matrix(y_Truth, y_Predicted,labels=clf_classes) 
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=labels)
    disp.plot()
    plt.xticks(rotation=90)
    plt.title(title)
    plt.show()

plotCM(y_test, y_pred,EXERCISE_DF[EXERCISE_DF.index.isin(svc.classes_)]["value"],svc.classes_,"Support Vector Classifier")

In [ ]:
def print_report(y_test, y_SVC,title):
    report = classification_report(y_test, y_SVC, output_dict=True)
    df = pd.DataFrame(report).transpose()
    EXERCISE_DF["key"] == EXERCISE_DF["key"].astype(float)
    idx_name = [ str(EXERCISE_DF[EXERCISE_DF.key==float(x)]["value"].values[0]) for x in df.index[:-3] ]

    for x in df.index[-3:]:
        idx_name.append(x)

    df["label"] = idx_name
    df.set_index("label", inplace=True)
    print(title)
    s1= df.iloc[:-3].style.format({'precision': '{:.2%}', 'recall': '{:.2%}', 'f1-score': '{:.2%}', 'recall': '{:.2%}','support': '{:,.0f}'})
    s1.background_gradient(cmap='YlOrRd',subset=df.columns[:-1] , axis=0) 
    display(s1)
    s2= df.iloc[-3:].style.format({'precision': '{:.2%}', 'recall': '{:.2%}', 'f1-score': '{:.2%}', 'recall': '{:.2%}','support': '{:,.0f}'})
    s2.background_gradient(cmap='YlOrRd',subset=df.columns[:-1] , axis=1) 
    display(s2)
print_report(y_test, y_pred, "Support Vector Machine")

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from numpy import interp
from sklearn.metrics import roc_auc_score, auc
from itertools import cycle

y = label_binarize(df.target, classes=list(set(df.target)))
X = df[feature_cols].to_numpy()
n_classes = y.shape[1]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X , y, stratify=y, test_size=.3, random_state=42)

# Does not work with rbf!!!!
classifier = OneVsRestClassifier(SVC(kernel='rbf', C=80, gamma=0.001, probability=True, random_state=42))
y_score_p = classifier.fit(X_train_p, y_train_p).decision_function(X_test_p)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

fig = plt.figure(figsize=[10, 5])

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_p[:, i], y_score_p[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['aqua', 'darkorange', 'cornflowerblue'])
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'
             ''.format( exercise_tuple[list(set(df.target))[i]].value , roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('One vs Rest multi-class ROC AUC')
fig.legend(loc="outside upper right",bbox_to_anchor=(0, 1))

In [ ]:
# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_p.ravel(), y_score_p.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Compute macro-average ROC curve and ROC area

# First aggregate all false positive rates
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

# Then interpolate all ROC curves at this points
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += interp(all_fpr, fpr[i], tpr[i])

# Finally average it and compute AUC
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot all ROC macro curves
plt.plot(fpr["macro"], tpr["macro"], label='macro-average ROC curve (area = {0:0.2f})'''.format(roc_auc["macro"]),
         color='navy', linewidth=2)

plt.plot(fpr["micro"], tpr["micro"], color='darkorange', lw=2, label='micro ROC curve (area = %0.2f)' % roc_auc["micro"])
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Stratified Data')
plt.legend(loc="lower right")
plt.show()


In [ ]:
from sklearn.metrics import cohen_kappa_score
svc = SVC(kernel="rbf",class_weight="balanced",C=76,gamma=0.005)
svc.fit(X_train_scaled,y_train)
y_pred = svc.predict(X_test_scaled)

print('kappa score', cohen_kappa_score(y_test, y_pred))
confusion = confusion_matrix(y_test, y_pred)
# sns.heatmap(confusion)
print(confusion)
print(classification_report(y_test, y_pred))


In [ ]:
# run lines in terminal to convert into html
print(f"cas_gpu_venv\\Scripts\\activate.bat")
print(f"jupyter-nbconvert --to html \"0_Module\\M3_Machine_Learning\\Project\\notebooks\\{os.path.basename(globals()['__vsc_ipynb_file__'])}\"")